# Track 2 — Ingeniería de Datos
**Rol:** CRB_DATA_ANALYTICS | **Tiempo:** 15 min | **Criterio:** Pipeline declarativo, calidad, observabilidad, CI/CD, linaje

In [ ]:
USE ROLE CRB_DATA_ANALYTICS;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Evidencia: Pipeline declarativo sin Spark

In [ ]:
-- Dynamic Tables: pipeline bronce→plata→oro SIN orquestador
SHOW DYNAMIC TABLES IN DATABASE CREDIBANCO_HOL;

In [ ]:
-- Ver datos transformados en la capa Silver
SELECT COUNT(*) AS filas_silver FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER;
SELECT * FROM CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER LIMIT 5;

In [ ]:
-- Linaje nativo: trazar la cadena end-to-end
SELECT * FROM TABLE(SNOWFLAKE.CORE.GET_LINEAGE(
  'CREDIBANCO_HOL.PAGOS.DT_AUTORIZACIONES_SILVER_USER', 'table', 'upstream', 3
));

## Bloque 2 — Ejecutar: Crear nueva DT + ver propagación

In [ ]:
-- Crear una Dynamic Table nueva: agregados por hora
CREATE OR REPLACE DYNAMIC TABLE CREDIBANCO_HOL.PAGOS.DT_HOURLY_<TU_USUARIO>
  TARGET_LAG = '5 minutes'
  WAREHOUSE = CREDIBANCO_HOL_WH
AS
SELECT DATE_TRUNC('hour', FECHA_HORA) AS hora,
       CIUDAD, COUNT(*) AS num_tx, SUM(MONTO) AS monto_total
FROM CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
GROUP BY 1, 2;

In [ ]:
-- Verificar refresh automático
SELECT name, refresh_mode, scheduling_state
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLES())
WHERE name LIKE 'DT_HOURLY%';

In [ ]:
-- Insertar dato nuevo y ver propagación
INSERT INTO CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
SELECT 999999, 1, 1, '4111111111111111', '5411', 'Bogota',
       CURRENT_TIMESTAMP(), 9999999, '00', 'ECOMMERCE';

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Crea un pipeline que lea la TRM de Colombia desde la API pública de la Superfinanciera (https://www.datos.gov.co/resource/mcec-87by.json) y la almacene en una tabla con un Task diario a las 8AM. Incluye Network Rule, External Access Integration, UDF Python y verificación.**

In [ ]:
-- Verificación final
SELECT 'T2_COMPLETO' AS status,
  (SELECT COUNT(*) FROM CREDIBANCO_HOL.PAGOS.DT_HOURLY_<TU_USUARIO>) AS filas_nueva_dt;